In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import re
import sqlite3
from urllib.parse import urljoin

class EStatIntegratedCollector:
    """e-Statファイルダウンロード、月別シート解析、DB保存を統合したクラス"""
    
    def __init__(self, db_path='estat_integrated_data.db'):
        self.base_url = "https://www.e-stat.go.jp"
        self.db_path = db_path
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
        self.crawl_delay = 3.14
        self.init_database()

    def init_database(self):
        """DBの疎通確認"""
        conn = sqlite3.connect(self.db_path)
        conn.close()
        print(f"✓ データベース準備完了: {self.db_path}")

    def _safe_table_name(self, name):
        """テーブル名として使えない文字を置換"""
        return re.sub(r'[^a-zA-Z0-9_]', '_', name)

    def download_and_process(self, stat_inf_id, table_patterns=['第1表'], 
                            output_dir='processed_data', 
                            save_to_db=True):
        """
        指定されたstatInfIdのExcelをダウンロードし、
        特定パターンのシートを解析してDBおよびCSVに保存する
        """
        os.makedirs(output_dir, exist_ok=True)
        temp_file = f"temp_{stat_inf_id}.xlsx"
        
        try:
            # 1. ダウンロード
            download_url = f"{self.base_url}/stat-search/file-download?statInfId={stat_inf_id}&fileKind=0"
            print(f"📥 ダウンロード中: {stat_inf_id}")
            
            response = self.session.get(download_url, timeout=60)
            response.raise_for_status()
            with open(temp_file, 'wb') as f:
                f.write(response.content)

            # 2. エンジン判定と全シート取得
            engine = 'xlrd' if response.content[:8] == b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1' else 'openpyxl'
            excel_file = pd.ExcelFile(temp_file, engine=engine)
            all_sheets = excel_file.sheet_names

            # 3. パターンごとに処理
            for pattern in table_patterns:
                matched_sheets = [s for s in all_sheets if pattern in s]
                if not matched_sheets:
                    print(f"⚠ パターン '{pattern}' に一致するシートはありません。")
                    continue

                # 月順にソート
                matched_sheets.sort(key=lambda s: int(re.search(r'(\d+)月', s).group(1)) if re.search(r'(\d+)月', s) else 999)
                
                all_df_list = []
                print(f"\n📊 {pattern} の解析開始 ({len(matched_sheets)}枚)")

                for sheet_name in matched_sheets:
                    # ヘッダーなしで読み込み（後でクリーニングが必要な場合が多いため）
                    df = pd.read_excel(temp_file, sheet_name=sheet_name, engine=engine, header=None)
                    
                    # 月情報の付与
                    month_match = re.search(r'(\d+)月', sheet_name)
                    df.insert(0, 'source_sheet', sheet_name)
                    df.insert(1, 'month_val', int(month_match.group(1)) if month_match else 0)
                    
                    all_df_list.append(df)
                    print(f"  ✓ {sheet_name} 読み込み完了")

                if all_df_list:
                    combined_df = pd.concat(all_df_list, ignore_index=True)
                    
                    # CSV出力
                    safe_name = self._safe_table_name(pattern)
                    csv_path = os.path.join(output_dir, f"{stat_inf_id}_{safe_name}.csv")
                    combined_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
                    
                    # DB保存
                    if save_to_db:
                        self.save_dataframe_to_db(combined_df, safe_name)
                        print(f"  💾 DB保存完了 (テーブル: {safe_name})")
                    
                    print(f"  📝 CSV保存完了: {csv_path}")

        finally:
            if os.path.exists(temp_file):
                os.remove(temp_file)
                print(f"\n🧹 一時ファイルを削除しました。")

    def save_dataframe_to_db(self, df, table_name):
        """DataFrameをSQLiteに保存（既存データは置換）"""
        # カラム名を文字列化（SQLエラー防止）
        df.columns = [str(c) for c in df.columns]
        conn = sqlite3.connect(self.db_path)
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        conn.close()

# --- 実行セクション ---
if __name__ == '__main__':
    collector = EStatIntegratedCollector()
    
    # ターゲットの統計ID (例: 鉄道統計など)
    target_id = '000040288348' 
    
    print("="*60)
    print("🚀 e-Stat 統合収集システム起動")
    print("="*60)
    
    # 複数の表を一度にDB化・CSV化
    collector.download_and_process(
        stat_inf_id=target_id,
        table_patterns=['第1表', '第2表'],
        output_dir='railway_monthly_data',
        save_to_db=True
    )
    
    print("\n✅ 全工程が完了しました。")

✓ データベース準備完了: estat_integrated_data.db
🚀 e-Stat 統合収集システム起動
📥 ダウンロード中: 000040288348

📊 第1表 の解析開始 (25枚)
  ✓ 第1表(1月) 読み込み完了
  ✓ 参考第1表(1月) 読み込み完了
  ✓ 第1表(2月) 読み込み完了
  ✓ 参考第1表(2月) 読み込み完了
  ✓ 第1表(3月) 読み込み完了
  ✓ 参考第1表(3月) 読み込み完了
  ✓ 第1表(4月) 読み込み完了
  ✓ 参考第1表(4月) 読み込み完了
  ✓ 第1表(5月) 読み込み完了
  ✓ 参考第1表(5月) 読み込み完了
  ✓ 第1表(6月) 読み込み完了
  ✓ 参考第1表(6月) 読み込み完了
  ✓ 第1表(7月) 読み込み完了
  ✓ 参考第1表(7月) 読み込み完了
  ✓ 第1表(8月) 読み込み完了
  ✓ 参考第1表(8月) 読み込み完了
  ✓ 第1表(9月) 読み込み完了
  ✓ 参考第1表(9月) 読み込み完了
  ✓ 第1表(10月) 読み込み完了
  ✓ 参考第1表(10月) 読み込み完了
  ✓ 第1表(11月) 読み込み完了
  ✓ 参考第1表(11月) 読み込み完了
  ✓ 第1表(12月) 読み込み完了
  ✓ 参考第1表(12月) 読み込み完了
  ✓ 参考第1表(年計) 読み込み完了
  💾 DB保存完了 (テーブル: _1_)
  📝 CSV保存完了: railway_monthly_data/000040288348__1_.csv

📊 第2表 の解析開始 (25枚)
  ✓ 第2表(1月) 読み込み完了
  ✓ 参考第2表(1月) 読み込み完了
  ✓ 第2表(2月) 読み込み完了
  ✓ 参考第2表(2月) 読み込み完了
  ✓ 第2表(3月) 読み込み完了
  ✓ 参考第2表(3月) 読み込み完了
  ✓ 第2表(4月) 読み込み完了
  ✓ 参考第2表(4月) 読み込み完了
  ✓ 第2表(5月) 読み込み完了
  ✓ 参考第2表(5月) 読み込み完了
  ✓ 第2表(6月) 読み込み完了
  ✓ 参考第2表(6月) 読み込み完了
  ✓ 第2表(7月) 読み込み完了
  ✓ 参考第2表(7月) 読み込み完了
  ✓ 第2表(8月) 読み